In [10]:
import pandas as pd
import numpy as np
import simpy
import joblib
import os
from scipy import stats

In [2]:
model = joblib.load('../model/xgboost_scheduler_model.pkl')

#### HPC Job Class

In [3]:
class HPCJob:
    """Represents a single job moving through the simulation."""
    def __init__(self, env, job_id, req_cpus, req_time, actual_runtime, queue_num, think_time):
        self.env = env
        self.job_id = job_id
        
        self.req_cpus = req_cpus
        self.req_time = req_time
        self.actual_runtime = actual_runtime  
        self.queue_num = queue_num
        self.think_time = think_time
        
        self.preceding_job_number = 0
        self.current_cpu_utilization = 0
        
        self.submit_time = env.now
        self.start_time = None
        self.end_time = None
        self.predicted_wait_time = None 

    def get_features(self):
        """Packages the current state into the exact format XGBoost expects."""
        return pd.DataFrame([{
            'requested_processors': self.req_cpus,
            'requested_time': self.req_time,
            'queue_number': self.queue_num,
            'think_time': self.think_time,
            'preceding_job_number': self.preceding_job_number,
            'current_cpu_utilization': self.current_cpu_utilization
        }])

### The HPC Cluster

In [4]:
class HPCCluster:
    def __init__(self, env, total_cpus, scheduler_policy='FCFS', ai_model=None):
        self.env = env
        self.total_cpus = total_cpus
        self.cpus_available = total_cpus
        
        self.scheduler_policy = scheduler_policy
        self.ai_model = ai_model
        
        self.queue = []          
        self.running_jobs = []   
        self.completed_jobs = [] 
        
        self.schedule_trigger = env.event()
        self.env.process(self.scheduler_loop())

    def wake_up_scheduler(self):
        if not self.schedule_trigger.triggered:
            self.schedule_trigger.succeed()

    def submit_job(self, job):
        self.queue.append(job)
        self.wake_up_scheduler()

    def execute_job(self, job):
        self.cpus_available -= job.req_cpus
        job.start_time = self.env.now
        self.running_jobs.append(job)
        
        # The job runs for its SECRET actual time, which the scheduler doesn't know!
        yield self.env.timeout(job.actual_runtime)
        
        job.end_time = self.env.now
        self.cpus_available += job.req_cpus
        self.running_jobs.remove(job)
        self.completed_jobs.append(job)
        self.wake_up_scheduler()

    def scheduler_loop(self):
        while True:
            yield self.schedule_trigger
            self.schedule_trigger = self.env.event() 
            
            if not self.queue:
                continue

            # --- STEP 1: SORT THE QUEUE ---
            if self.scheduler_policy == 'FCFS':
                self.queue.sort(key=lambda j: j.submit_time)
                
            elif self.scheduler_policy == 'XGBOOST':
                # THE BATCH INFERENCE UPGRADE
                current_util = self.total_cpus - self.cpus_available
                q_len = len(self.queue)
                
                # 1. Build a raw Python list of dictionaries (Incredibly fast)
                features_list = []
                for job in self.queue:
                    job.preceding_job_number = q_len
                    job.current_cpu_utilization = current_util
                    features_list.append({
                        'requested_processors': job.req_cpus,
                        'requested_time': job.req_time,
                        'queue_number': job.queue_num,
                        'think_time': job.think_time,
                        'preceding_job_number': q_len,
                        'current_cpu_utilization': current_util
                    })
                
                # 2. Convert to Pandas ONCE and Batch Predict
                batch_df = pd.DataFrame(features_list)
                batch_predictions = self.ai_model.predict(batch_df)
                
                # 3. Assign the scores back to the objects
                for i, job in enumerate(self.queue):
                    job.predicted_wait_time = batch_predictions[i]
                
                # 4. Sort the queue
                self.queue.sort(key=lambda j: j.predicted_wait_time)

            # --- STEP 2: TRUE EASY BACKFILLING ---
            jobs_to_evaluate = list(self.queue)
            highest_priority_job = jobs_to_evaluate[0]

            if highest_priority_job.req_cpus <= self.cpus_available:
                self.queue.remove(highest_priority_job)
                self.env.process(self.execute_job(highest_priority_job))
                self.wake_up_scheduler() 
                continue

            projected_freed_cpus = self.cpus_available
            shadow_time = self.env.now
            
            running_sorted = sorted(self.running_jobs, key=lambda j: j.start_time + j.req_time)
            
            for running_job in running_sorted:
                projected_freed_cpus += running_job.req_cpus
                shadow_time = running_job.start_time + running_job.req_time
                if projected_freed_cpus >= highest_priority_job.req_cpus:
                    break 

            for backfill_job in jobs_to_evaluate[1:]:
                if backfill_job.req_cpus <= self.cpus_available:
                    expected_end_time = self.env.now + backfill_job.req_time
                    if expected_end_time <= shadow_time:
                        self.queue.remove(backfill_job)
                        self.env.process(self.execute_job(backfill_job))
                        self.wake_up_scheduler()
                        break

### Metrics Evaluation

In [5]:
def calculate_metrics(completed_jobs, total_cpus, simulation_end_time):
    """Calculates Wait Time, BSLD, and Cluster Utilization."""
    wait_times = []
    bslds = []
    total_cpu_seconds_used = 0
    
    for job in completed_jobs:
        wait_time = job.start_time - job.submit_time
        wait_times.append(wait_time)
        
        # Bounded Slowdown (BSLD)
        # We use a 10-second lower bound to prevent tiny jobs from breaking the math
        bounded_runtime = max(10, job.actual_runtime) 
        bsld = max(1, (wait_time + bounded_runtime) / bounded_runtime)
        bslds.append(bsld)
        
        # CPU-Seconds = CPUs requested * Actual time spent computing
        total_cpu_seconds_used += (job.req_cpus * job.actual_runtime)
        
    avg_wait = np.mean(wait_times)
    avg_bsld = np.mean(bslds)
    
    # Utilization = (Compute Time) / (Total Capacity of the Cluster over Time)
    max_possible_cpu_seconds = total_cpus * simulation_end_time
    utilization = (total_cpu_seconds_used / max_possible_cpu_seconds) * 100
    
    return avg_wait, avg_bsld, utilization

### The Simulation

In [6]:
def run_simulation(workload_df, scheduler_policy, ai_model=None):
    """Wraps the entire universe into a single callable run."""
    env = simpy.Environment()
    cluster = HPCCluster(env, total_cpus=1920, scheduler_policy=scheduler_policy, ai_model=ai_model)
    
    def job_spawner(env, cluster, df):
        for _, row in df.iterrows():
            time_to_wait = row['submit_time'] - env.now
            if time_to_wait > 0:
                yield env.timeout(time_to_wait)
                
            job = HPCJob(
                env=env, job_id=_, req_cpus=row['requested_processors'],
                req_time=row['requested_time'], actual_runtime=row['actual_runtime'],
                queue_num=row['queue_number'], think_time=row['think_time']
            )
            cluster.submit_job(job)

    env.process(job_spawner(env, cluster, workload_df))
    env.run() 
    
    avg_wait, avg_bsld, util = calculate_metrics(cluster.completed_jobs, cluster.total_cpus, env.now)
    
    return {
        "Policy": scheduler_policy,
        "Jobs Processed": len(cluster.completed_jobs),
        "Sim Duration (Days)": env.now / 86400,
        "Avg Wait Time (Hrs)": avg_wait / 3600,
        "Avg BSLD": avg_bsld,
        "Utilization (%)": util
    }

In [7]:
print("Loading Model and Workload 01...")
xgb_model = joblib.load('../model/xgboost_scheduler_model.pkl')
workload = pd.read_csv('../synthetic_workloads/workload_01.csv')

Loading Model and Workload 01...


### Test Simulation on Worload 1

In [8]:
print("\n--- Running Baseline: Standard FCFS EASY ---")
fcfs_results = run_simulation(workload, scheduler_policy='FCFS')
for key, val in fcfs_results.items():
    print(f"{key}: {val:.2f}" if isinstance(val, float) else f"{key}: {val}")


--- Running Baseline: Standard FCFS EASY ---
Policy: FCFS
Jobs Processed: 10000
Sim Duration (Days): 4.73
Avg Wait Time (Hrs): 6.83
Avg BSLD: 6.45
Utilization (%): 63.12


In [9]:
print("\n--- Running Proposed: ML-Guided EASY ---")
ml_results = run_simulation(workload, scheduler_policy='XGBOOST', ai_model=xgb_model)
for key, val in ml_results.items():
    print(f"{key}: {val:.2f}" if isinstance(val, float) else f"{key}: {val}")


--- Running Proposed: ML-Guided EASY ---
Policy: XGBOOST
Jobs Processed: 10000
Sim Duration (Days): 4.28
Avg Wait Time (Hrs): 4.15
Avg BSLD: 2.89
Utilization (%): 69.71


### Final Simulation

In [11]:
NUM_SIMULATIONS = 50
master_results = []

In [14]:
for i in range(1, NUM_SIMULATIONS + 1):
    # 1. Load the specific stochastic reality
    file_path = f'../synthetic_workloads/workload_{i:02d}.csv'
    workload_df = pd.read_csv(file_path)
    
    print(f"Processing Workload {i:02d}/{NUM_SIMULATIONS}...")
    
    # 2. Run Track A (Baseline)
    fcfs_metrics = run_simulation(workload_df, scheduler_policy='FCFS')
    
    # 3. Run Track B (Proposed)
    xgb_metrics = run_simulation(workload_df, scheduler_policy='XGBOOST', ai_model=xgb_model)
    
    # 4. Log the comparative metrics for this specific timeline
    master_results.append({
        'Iteration': i,
        'FCFS_Wait_Hrs': fcfs_metrics['Avg Wait Time (Hrs)'],
        'XGB_Wait_Hrs': xgb_metrics['Avg Wait Time (Hrs)'],
        'FCFS_BSLD': fcfs_metrics['Avg BSLD'],
        'XGB_BSLD': xgb_metrics['Avg BSLD'],
        'FCFS_Util_%': fcfs_metrics['Utilization (%)'],
        'XGB_Util_%': xgb_metrics['Utilization (%)']
    })

Processing Workload 01/50...
Processing Workload 02/50...
Processing Workload 03/50...
Processing Workload 04/50...
Processing Workload 05/50...
Processing Workload 06/50...
Processing Workload 07/50...
Processing Workload 08/50...
Processing Workload 09/50...
Processing Workload 10/50...
Processing Workload 11/50...
Processing Workload 12/50...
Processing Workload 13/50...
Processing Workload 14/50...
Processing Workload 15/50...
Processing Workload 16/50...
Processing Workload 17/50...
Processing Workload 18/50...
Processing Workload 19/50...
Processing Workload 20/50...
Processing Workload 21/50...
Processing Workload 22/50...
Processing Workload 23/50...
Processing Workload 24/50...
Processing Workload 25/50...
Processing Workload 26/50...
Processing Workload 27/50...
Processing Workload 28/50...
Processing Workload 29/50...
Processing Workload 30/50...
Processing Workload 31/50...
Processing Workload 32/50...
Processing Workload 33/50...
Processing Workload 34/50...
Processing Wor

In [15]:
results_df = pd.DataFrame(master_results)
results_df.to_csv('final_monte_carlo_results.csv', index=False)

In [17]:
print("\n=== FINAL GRAND AVERAGES (50 Workloads) ===")
print(f"FCFS Wait Time: {results_df['FCFS_Wait_Hrs'].mean():.2f} hrs | XGB Wait Time: {results_df['XGB_Wait_Hrs'].mean():.2f} hrs")
print(f"FCFS BSLD:      {results_df['FCFS_BSLD'].mean():.2f}     | XGB BSLD:      {results_df['XGB_BSLD'].mean():.2f}")
print(f"FCFS Util:      {results_df['FCFS_Util_%'].mean():.2f}%   | XGB Util:      {results_df['XGB_Util_%'].mean():.2f}%")


=== FINAL GRAND AVERAGES (50 Workloads) ===
FCFS Wait Time: 6.14 hrs | XGB Wait Time: 3.58 hrs
FCFS BSLD:      6.02     | XGB BSLD:      2.75
FCFS Util:      59.16%   | XGB Util:      66.07%


In [18]:
t_stat, p_value = stats.ttest_rel(results_df['FCFS_Wait_Hrs'], results_df['XGB_Wait_Hrs'])

print("\n=== STATISTICAL SIGNIFICANCE (Wait Time) ===")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value:     {p_value:.4e}")


=== STATISTICAL SIGNIFICANCE (Wait Time) ===
T-Statistic: 19.4535
P-Value:     1.0797e-24


In [19]:
if p_value < 0.05:
    if results_df['XGB_Wait_Hrs'].mean() < results_df['FCFS_Wait_Hrs'].mean():
        print("XGBoost is statistically significantly BETTER than standard FCFS.")
    else:
        print("WARNING: XGBoost is statistically significantly WORSE than FCFS.")
else:
    print("TIE: There is no statistically significant difference between the two algorithms.")

XGBoost is statistically significantly BETTER than standard FCFS.
